# 04 — Comprendre le dialogue avec Claude Code

**Module 1.1 — Premiers pas avec Claude Code**

---

Ce notebook décrit le fonctionnement interne de Claude Code : ce qui se passe entre le moment où une instruction est soumise et celui où le résultat apparaît. Comprendre ce mécanisme permet de formuler de meilleures instructions, d'identifier les points de friction et de savoir quand intervenir.

**Durée estimée :** 30 minutes  
**Prérequis :** Notebooks 01, 02, 03

---

## 1. Pourquoi comprendre le processus

Il est tentant d'utiliser Claude Code comme une boîte noire : on formule une demande, on attend le résultat. Cette approche fonctionne pour les tâches simples. Elle devient insuffisante dès que :

- le résultat ne correspond pas à ce qui était attendu
- l'exécution s'arrête à mi-chemin
- Claude modifie des fichiers qui n'auraient pas dû l'être
- on ne sait pas si le résultat est fiable

### Trois raisons de comprendre le processus

**1. Ne pas être dépendant**  
Un utilisateur qui comprend les phases d'exécution peut relancer une étape précise, corriger une instruction intermédiaire ou déléguer différemment selon la complexité de la tâche.

**2. Savoir quand intervenir**  
Certaines phases nécessitent une validation humaine : écriture dans des fichiers critiques, exécution de commandes système, accès à des données sensibles. Sans connaissance du processus, on ne sait pas quand lever la main.

**3. Améliorer ses instructions**  
La qualité de l'output dépend directement de la qualité de l'input. Comprendre ce que Claude fait à chaque étape permet de lui fournir exactement ce dont il a besoin — ni plus, ni moins.

---

## 2. Les 5 phases d'exécution

Lorsqu'une instruction est soumise à Claude Code, l'exécution suit systématiquement cinq phases. Elles peuvent se chevaucher ou se répéter sur les tâches longues, mais la séquence de base reste la même.

```
Instruction
    │
    ▼
Phase 1 : Analyse (planning)
    │
    ▼
Phase 2 : Lecture du contexte
    │
    ▼
Phase 3 : Écriture / modification
    │
    ▼
Phase 4 : Exécution
    │
    ▼
Phase 5 : Synthèse et vérification
    │
    ▼
Résultat
```

### Phase 1 — Analyse de l'instruction (planning)

**Ce qui se passe**  
Claude décompose l'instruction en sous-tâches ordonnées. Il identifie les ambiguïtés, les dépendances entre étapes et les informations manquantes. Si l'instruction est suffisamment claire, il passe directement à la phase suivante. Sinon, il demande des précisions.

**Outils utilisés**  
Aucun outil externe à ce stade — c'est un traitement interne.

**Ce qu'il faut surveiller**  
- Claude reformule-t-il correctement l'objectif ?
- Les sous-tâches identifiées correspondent-elles à ce qui était voulu ?
- Pose-t-il des questions pertinentes ou des questions inutiles ?

**Exemple concret**

Instruction : *"Analyse les données de consommation et génère un rapport PDF avec les 3 indicateurs clés."*

Planning interne :
1. Localiser le fichier de données
2. Charger et explorer les données
3. Calculer les 3 indicateurs (lesquels ? → question)
4. Générer les graphiques
5. Produire le PDF

Claude identifie l'ambiguïté "3 indicateurs clés" et peut soit demander, soit faire un choix en le documentant.

### Phase 2 — Lecture du contexte (Read, Glob, Grep)

**Ce qui se passe**  
Claude explore le système de fichiers pour comprendre la structure du projet. Il lit les fichiers pertinents, cherche des patterns dans le code existant, identifie les dépendances et les conventions déjà en place.

**Outils utilisés**

| Outil | Rôle |
|-------|------|
| `Read` | Lit le contenu d'un fichier spécifique |
| `Glob` | Liste les fichiers correspondant à un pattern (`*.csv`, `**/*.py`) |
| `Grep` | Cherche un pattern dans le contenu des fichiers |

**Ce qu'il faut surveiller**  
- Quels fichiers Claude consulte-t-il ? Sont-ils pertinents ?
- Lit-il des fichiers sensibles (credentials, config avec mots de passe) ?
- La lecture est-elle exhaustive ou superficielle ?

**Exemple concret**

Pour analyser `consommation_energie.csv`, Claude va :
- `Glob`: chercher `*.csv` dans le répertoire
- `Read`: lire les premières lignes du CSV pour en comprendre la structure
- `Read`: consulter `CLAUDE.md` s'il existe pour les conventions du projet
- `Grep`: chercher d'éventuels scripts existants sur le même dataset

### Phase 3 — Écriture / modification (Write, Edit)

**Ce qui se passe**  
Claude produit le code, les fichiers ou les modifications demandées. Il utilise les informations collectées en phase 2 pour respecter les conventions existantes et éviter les conflits.

**Outils utilisés**

| Outil | Rôle |
|-------|------|
| `Write` | Crée un nouveau fichier ou écrase un fichier existant |
| `Edit` | Modifie une portion précise d'un fichier existant |

**Ce qu'il faut surveiller**  
- Claude crée-t-il les bons fichiers aux bons endroits ?
- Utilise-t-il `Edit` (modification ciblée) plutôt que `Write` (réécriture complète) sur des fichiers existants ?
- Le code généré respecte-t-il les conventions du projet ?

**Exemple concret**

Génération d'un script d'analyse :
- `Write`: crée `analyse_energie.py` avec le code complet
- `Edit`: ajoute une fonction manquante dans un script existant sans toucher au reste

Différence importante : `Write` écrase — il faut s'assurer qu'on ne perd pas de travail existant.

### Phase 4 — Exécution (Bash)

**Ce qui se passe**  
Claude exécute des commandes dans le terminal : lancer des scripts Python, installer des packages, déplacer des fichiers, lancer des tests. C'est la phase avec le plus d'impact sur le système.

**Outils utilisés**

| Outil | Rôle |
|-------|------|
| `Bash` | Exécute une commande shell et retourne stdout/stderr |

**Ce qu'il faut surveiller**  
- Quelles commandes sont exécutées ? Sont-elles réversibles ?
- Les commandes `rm`, `mv`, `pip install` méritent une attention particulière
- Les erreurs de la commande sont-elles traitées ?

**Exemple concret**

```bash
# Claude peut exécuter ces commandes :
python analyse_energie.py          # lancer le script
pip install plotly                 # installer un package
cp rapport.html ./output/          # déplacer un fichier
```

La commande `rm -rf` (suppression définitive) doit toujours déclencher une validation manuelle.

### Phase 5 — Synthèse et vérification

**Ce qui se passe**  
Claude vérifie que le résultat correspond à l'instruction initiale. Il relit les fichiers produits, analyse les sorties d'exécution, identifie les écarts et les documente. Il produit ensuite un résumé de ce qui a été fait.

**Outils utilisés**  
`Read` pour relire les fichiers produits, `Bash` pour des vérifications (ex: `python -c "import module"`).

**Ce qu'il faut surveiller**  
- Le résumé final est-il fidèle à ce qui a été fait ?
- Claude signale-t-il clairement ce qu'il n'a pas pu faire ?
- Y a-t-il des avertissements ou des limitations à noter ?

**Exemple concret**

Après génération du rapport :
- `Read`: relit `rapport.html` pour vérifier la structure
- Constate que le graphique 3 est vide faute de données suffisantes
- Signale le problème dans le résumé avec une suggestion de correction

---

## 3. Les outils en détail

Chaque outil a un périmètre précis. Les confondre est une source fréquente d'instructions mal calibrées.

### Outil : Read

**Ce qu'il fait**  
Lit le contenu d'un fichier et le retourne dans le contexte de Claude. Fonctionne avec les fichiers texte, Python, CSV, JSON, PDF, images.

**Quand il est utilisé**  
- Comprendre la structure d'un fichier existant avant de le modifier
- Lire les données avant d'écrire le code qui les traite
- Vérifier le résultat après une modification

**Exemple**
```
Input  : Read("/data/consommation.csv")
Output : date,batiment,zone,kwh\n2024-01-01,Usine A,Production,4850\n...
```

**Pièges courants**  
- Lire un fichier très volumineux ralentit l'exécution (Claude lit par blocs)
- Les fichiers binaires (xlsx, images) sont lus différemment — ne pas supposer que Claude voit le contenu Excel formaté

### Outil : Write

**Ce qu'il fait**  
Crée un fichier avec le contenu fourni. Si le fichier existe déjà, il est **entièrement remplacé**.

**Quand il est utilisé**  
- Créer un nouveau script Python
- Générer un fichier de configuration
- Produire un rapport HTML ou Markdown

**Exemple**
```
Input  : Write("/scripts/analyse.py", contenu_complet)
Output : Fichier créé (2847 bytes)
```

**Pièges courants**  
- Ne jamais utiliser sur un fichier qui contient du travail non sauvegardé
- Préférer `Edit` pour des modifications ponctuelles d'un fichier existant

### Outil : Edit

**Ce qu'il fait**  
Remplace une portion de texte précise dans un fichier existant, sans toucher au reste. Utilise une correspondance exacte sur la chaîne à remplacer.

**Quand il est utilisé**  
- Corriger une fonction dans un script existant
- Ajouter une ligne dans un fichier de configuration
- Renommer une variable dans tout un fichier

**Exemple**
```
Input  : Edit(fichier, old="df.fillna(0)", new="df.fillna(df.mean())")
Output : Modification appliquée (ligne 47)
```

**Pièges courants**  
- Si le texte à remplacer n'est pas unique dans le fichier, l'outil échoue
- Claude doit avoir lu le fichier (`Read`) avant de pouvoir l'éditer

### Outil : Bash

**Ce qu'il fait**  
Exécute une commande shell et retourne stdout et stderr. Accès complet au système dans les limites des permissions configurées.

**Quand il est utilisé**  
- Lancer un script Python
- Installer des dépendances
- Vérifier qu'un module est installé
- Déplacer ou renommer des fichiers

**Exemple**
```
Input  : Bash("python scripts/analyse.py --input data/energie.csv")
Output : Rapport généré : output/rapport_2024.html\nTemps : 3.2s
```

**Pièges courants**  
- C'est l'outil avec le plus d'impact irréversible sur le système
- Une erreur dans une commande `rm` ou `mv` peut supprimer des données
- Toujours vérifier ce qui sera exécuté avant de valider

### Outil : Glob

**Ce qu'il fait**  
Liste les fichiers correspondant à un pattern (comme `*.csv` ou `**/*.py`). Retourne les chemins triés.

**Quand il est utilisé**  
- Découvrir la structure d'un projet avant de commencer
- Trouver tous les fichiers d'un certain type
- Vérifier qu'un fichier existe

**Exemple**
```
Input  : Glob("data/**/*.csv")
Output : data/module-1/consommation_energie.csv
         data/module-1/capteurs_temperature.csv
         data/module-2/ventes_2024.csv
```

**Pièges courants**  
- Un pattern trop large peut retourner des centaines de fichiers inutiles
- Ne donne pas le contenu des fichiers, seulement leurs chemins

### Outil : Grep

**Ce qu'il fait**  
Cherche un pattern (texte ou expression régulière) dans le contenu des fichiers. Retourne les lignes correspondantes avec leur numéro de ligne.

**Quand il est utilisé**  
- Trouver où une variable est définie dans un projet
- Vérifier si une bibliothèque est déjà importée
- Identifier toutes les occurrences d'une valeur dans les données

**Exemple**
```
Input  : Grep(pattern="import pandas", path="scripts/")
Output : scripts/analyse.py:3: import pandas as pd
         scripts/nettoyage.py:1: import pandas as pd
```

**Pièges courants**  
- Chercher un pattern trop générique (`import`) retourne trop de résultats
- Ne fonctionne que sur du texte — pas sur des fichiers binaires

---

## 4. Le système de permissions

Claude Code dispose d'un mécanisme de permissions qui contrôle quels outils peuvent s'exécuter automatiquement et lesquels requièrent une validation manuelle.

### Les 3 niveaux

| Niveau | Outils concernés | Comportement par défaut |
|--------|-----------------|------------------------|
| **Lecture** | `Read`, `Glob`, `Grep` | Autorisé automatiquement |
| **Écriture** | `Write`, `Edit` | Demande confirmation selon le contexte |
| **Exécution** | `Bash` | Demande confirmation selon la commande |

La lecture est généralement sans risque. L'écriture et l'exécution ont des effets sur le système qui peuvent être difficiles à inverser.

### Comment configurer les permissions

Les permissions se configurent dans le fichier `.claude/settings.json` à la racine du projet, ou via les options de la CLI.

```json
{
  "permissions": {
    "allow": [
      "Read",
      "Write",
      "Edit",
      "Bash(python *)",
      "Bash(pip install *)"
    ],
    "deny": [
      "Bash(rm *)",
      "Bash(sudo *)"
    ]
  }
}
```

Le mode `ask` déclenche une demande de confirmation interactive pour chaque utilisation d'un outil.

### Implications de sécurité

Trois risques principaux à connaître :

**1. Suppression accidentelle**  
Une commande `Bash(rm ...)` mal formulée peut supprimer des fichiers de données. Bloquer `rm -rf` par défaut est une bonne pratique.

**2. Accès aux données sensibles**  
Si Claude lit un fichier `.env` contenant des mots de passe ou des clés API, ces informations entrent dans son contexte. Ne pas avoir de données sensibles dans les répertoires de travail.

**3. Installation de packages non vérifiés**  
Autoriser `pip install *` sans restriction permet l'installation de n'importe quel package. Préférer une liste explicite de packages autorisés dans les environnements de production.

### Bonnes pratiques

- Commencer en mode restrictif, élargir au besoin
- Bloquer explicitement `rm`, `sudo`, `curl | bash`
- Travailler dans un environnement virtuel Python isolé
- Mettre les données sensibles hors du répertoire de travail

---

## 5. Optimiser ses instructions

La qualité d'une instruction détermine directement la qualité du résultat. Cinq paires d'exemples illustrent les différences concrètes.

### 5 paires : instruction faible vs instruction précise

**Pair 1 — Contexte manquant**

| | Instruction |
|--|-------------|
| Faible | *"Analyse les données."* |
| Précise | *"Analyse `data/consommation_energie.csv` (colonnes : date, batiment, zone, kwh). Calcule la consommation totale par bâtiment pour 2024."* |

La version faible force Claude à deviner le fichier, les colonnes et l'objectif.

---

**Pair 2 — Format de sortie non spécifié**

| | Instruction |
|--|-------------|
| Faible | *"Génère un rapport sur la consommation."* |
| Précise | *"Génère un rapport HTML dans `output/rapport_jan2024.html` avec : titre, tableau récapitulatif, 2 graphiques Plotly (bar + line), section recommandations."* |

Sans format défini, le résultat peut être un texte brut, un CSV, ou un notebook — rarement ce qui était attendu.

---

**Pair 3 — Contraintes absentes**

| | Instruction |
|--|-------------|
| Faible | *"Nettoie les données."* |
| Précise | *"Nettoie `data/brut.csv` : supprime les doublons, remplace les valeurs négatives de kwh par NaN, complète les NaN par la moyenne du groupe (batiment × mois). Sauvegarde dans `data/propre.csv`. Ne pas modifier le fichier original."* |

La version faible laisse Claude décider de ce que "nettoyer" signifie — avec des conséquences potentiellement irréversibles.

---

**Pair 4 — Objectif métier absent**

| | Instruction |
|--|-------------|
| Faible | *"Fais un graphique de la consommation."* |
| Précise | *"Crée un graphique en barres (Plotly) de la consommation totale kwh par bâtiment pour Q1 2024. Mettre en évidence le bâtiment au-dessus de la moyenne. Titre : 'Consommation Q1 2024 — Vue comparative'. Exporter en PNG dans `output/`."* |

L'objectif métier (identifier les outliers) guide le choix de mise en forme.

---

**Pair 5 — Périmètre non borné**

| | Instruction |
|--|-------------|
| Faible | *"Améliore le script d'analyse."* |
| Précise | *"Dans `scripts/analyse.py`, ajoute uniquement une fonction `calcul_baseline(df, col='kwh')` qui retourne la moyenne mensuelle par bâtiment. Ne pas modifier les fonctions existantes."* |

Sans périmètre, Claude peut réécrire l'ensemble du script selon ses préférences.

### Structure d'une bonne instruction

Un template en 4 blocs :

```
CONTEXTE   : [Situation, fichiers impliqués, état actuel]
DONNÉES    : [Source, colonnes clés, particularités connues]
OUTPUT     : [Format attendu, emplacement, structure]
CONTRAINTES: [Ce qu'il ne faut pas toucher, limites, règles métier]
```

### Exemple 1 — Rapport énergie

```
CONTEXTE    : Préparation d'un rapport mensuel pour le service énergie.
              Fichier : data/consommation_energie.csv
              Colonnes : date (YYYY-MM-DD), batiment, zone, type_energie, kwh

DONNÉES     : Données janvier–juin 2024. Quelques lignes avec kwh négatifs
              (erreurs de capteurs) à exclure de l'analyse.

OUTPUT      : Script Python `scripts/rapport_mensuel.py` qui génère
              `output/rapport_{mois}.html` avec :
              - Tableau : consommation totale par bâtiment
              - Graphique 1 : évolution mensuelle (line chart)
              - Graphique 2 : répartition électricité vs gaz (pie chart)

CONTRAINTES : Ne pas modifier les fichiers CSV source.
              Utiliser uniquement pandas et plotly.
              Commentaires en français.
```

### Exemple 2 — Analyse production

```
CONTEXTE    : L'équipe maintenance veut identifier les lignes de production
              dont le rendement a chuté sur les 30 derniers jours.
              Fichier : data/production_industrielle.csv
              Colonnes : timestamp, ligne, produit, quantite_produite,
                         quantite_conforme, temps_arret_min

DONNÉES     : Données horaires. Le taux de conformité se calcule :
              quantite_conforme / quantite_produite.
              Seuil d'alerte : taux < 0.95 sur 3 jours consécutifs.

OUTPUT      : Notebook `notebooks/analyse_rendement.ipynb` avec :
              - Cellule 1 : chargement et aperçu
              - Cellule 2 : calcul du taux de conformité
              - Cellule 3 : identification des lignes en alerte
              - Cellule 4 : graphique timeline par ligne
              - Cellule 5 : tableau de synthèse exporté en CSV

CONTRAINTES : Aucune installation de package supplémentaire.
              Le notebook doit s'exécuter de A à Z sans erreur.
```

### Exemple 3 — Détection d'anomalies capteurs

```
CONTEXTE    : Les capteurs de température d'une salle serveur envoient
              des relevés toutes les 5 minutes. Des pics inexpliqués
              apparaissent certains weekends.
              Fichier : data/capteurs_temperature.csv
              Colonnes : timestamp, capteur_id, temperature_c, humidite_pct

DONNÉES     : Données sur 6 mois. Température normale : 18–22°C.
              Anomalie définie comme : température > 25°C pendant > 15 min
              (3 relevés consécutifs).

OUTPUT      : Script `scripts/detection_anomalies.py` qui :
              1. Charge les données
              2. Identifie les épisodes d'anomalie (début, fin, durée, max temp)
              3. Exporte la liste dans `output/anomalies.csv`
              4. Génère un graphique timeline avec les anomalies surlignées

CONTRAINTES : Définir une fonction `detecter_anomalies(df, seuil, duree_min)`
              pour que le seuil soit configurable.
              Pas de bibliothèques ML — uniquement pandas.
```

---

## 6. CLAUDE.md : le fichier de configuration

### Qu'est-ce que c'est ?

`CLAUDE.md` est un fichier Markdown placé à la racine d'un projet. Claude Code le lit automatiquement à chaque session. Il sert à contextualiser le projet : conventions, structure des données, règles métier, contraintes techniques.

C'est l'équivalent d'un briefing permanent que Claude reçoit avant chaque instruction. Bien rédigé, il réduit considérablement le besoin de répéter le contexte dans chaque demande.

### Où le placer ?

```
projet/
├── CLAUDE.md          ← ici, à la racine
├── data/
├── scripts/
└── notebooks/
```

Il peut aussi exister un `CLAUDE.md` global dans `~/` qui s'applique à tous les projets.

### Exemple minimal pour un projet data

```markdown
# Mon Projet Énergie

## Contexte
Analyse de la consommation énergétique d'un site industriel (3 bâtiments).
Données issues du système de GTB (Gestion Technique du Bâtiment).

## Structure
- `data/brut/`    — CSV exportés depuis GTB, ne jamais modifier
- `data/propre/`  — CSV nettoyés, générés par les scripts
- `scripts/`      — Scripts Python d'analyse
- `output/`       — Rapports et graphiques générés

## Données principales
Fichier : `data/brut/consommation_energie.csv`  
Colonnes :
- `date` : format YYYY-MM-DD
- `batiment` : Usine_A | Usine_B | Bureau_Central | Entrepot_Nord
- `zone` : Production | Bureaux | Eclairage | Climatisation
- `type_energie` : electricite | gaz
- `kwh` : consommation en kWh (float, parfois négatif = erreur capteur)

## Règles
- Langage : Python 3.12, bibliothèques : pandas, plotly uniquement
- Les scripts doivent avoir des commentaires en français
- Ne jamais modifier les fichiers dans `data/brut/`
- Nommage : `scripts/analyse-{sujet}-{date}.py`

## Contexte métier
- Objectif de réduction : -15% de consommation électrique d'ici fin 2025
- Baseline de référence : consommation moyenne 2023
- Seuil d'alerte : +20% par rapport à la baseline sur un mois
```

### Comment enrichir progressivement le CLAUDE.md

Au fil du projet, ajouter :

**Règles métier découvertes**
```markdown
## Règles métier
- Les données du weekend ont une consommation ~30% plus basse (activité réduite)
- Le bâtiment Entrepot_Nord a été rénové en mars 2024 : comparaison avant/après à isoler
- Les données avant 2023-06-01 sont peu fiables (calibration des capteurs)
```

**Patterns de code validés**
```markdown
## Patterns approuvés
- Toujours parser les dates avec `pd.to_datetime(df['date'], format='%Y-%m-%d')`
- Groupby de référence : `df.groupby(['batiment', df['date'].dt.month])['kwh'].sum()`
```

**Ce que Claude ne doit pas faire**
```markdown
## Interdits
- Ne pas utiliser matplotlib (on utilise plotly)
- Ne pas créer de fichiers dans `data/brut/`
- Ne pas arrondir les valeurs kwh (précision nécessaire pour les calculs de coût)
```

---

## 7. Résumé

### Table des concepts

| Concept | Rôle | À retenir |
|---------|------|----------|
| Phase 1 : Analyse | Planning de la tâche | Vérifier que Claude comprend bien l'objectif |
| Phase 2 : Lecture | Collecte du contexte | Surveiller quels fichiers sont lus |
| Phase 3 : Écriture | Production du code/fichiers | `Edit` > `Write` sur fichiers existants |
| Phase 4 : Exécution | Lancement de commandes | La phase la plus risquée — valider les commandes |
| Phase 5 : Synthèse | Vérification et résumé | Lire le résumé attentivement |
| `Read` | Lecture de fichier | Sans risque, mais attention aux fichiers sensibles |
| `Write` | Création/remplacement | Irréversible — vérifier avant |
| `Edit` | Modification ciblée | Préférable à `Write` sur fichiers existants |
| `Bash` | Exécution shell | Risque maximal — toujours vérifier |
| `Glob` | Listing de fichiers | Sans risque |
| `Grep` | Recherche dans fichiers | Sans risque |
| Permissions | Contrôle des outils | Commencer restrictif, élargir au besoin |
| `CLAUDE.md` | Configuration du projet | Investissement une fois, bénéfice permanent |

### Arbre de décision : "Dois-je laisser Claude faire cela ?"

```
Claude propose une action
        │
        ▼
Est-ce une lecture ? (Read, Glob, Grep)
   OUI → Autoriser sans hésitation
   NON ↓
        │
        ▼
Est-ce une écriture ? (Write, Edit)
   OUI → Le fichier cible est-il critique ou contient-il du travail non sauvegardé ?
           OUI → Vérifier le contenu prévu avant de valider
           NON → Autoriser
   NON ↓
        │
        ▼
Est-ce une exécution ? (Bash)
   OUI → La commande est-elle réversible ?
           OUI (python, pip list...) → Autoriser
           NON (rm, mv, sudo...)     → Lire la commande complète avant de valider
```

---

**Prochain notebook** : 05 — Cas d'étude complet : analyse de consommation énergétique en 40 minutes